<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/01_Dataset_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# NOTEBOOK 01 — GOOGLE DRIVE MOUNT
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [2]:
# ============================================================
# 01.0 UPLOAD / VERIFY RAW DATASETS
# ============================================================

from pathlib import Path
from google.colab import files
import shutil

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

RAW_DIR = (
    PROJECT_ROOT /
    "data" /
    "raw"
)

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXPECTED_DATASETS = {
    "adult_income": [
        "adult_income.csv",
        "adult.csv",
    ],

    "bank_marketing": [
        "bank_marketing.csv",
        "bank.csv",
        "bank-full.csv",
    ],

    "diabetes_130us": [
        "diabetes_130us.csv",
        "diabetic_data.csv",
    ],
}

print("=" * 90)
print("RAW DATASET VERIFICATION")
print("=" * 90)

for dataset_id, filenames in EXPECTED_DATASETS.items():

    found = []

    for filename in filenames:

        path = RAW_DIR / filename

        if path.exists():
            found.append(path)

    if found:

        print(
            f"{dataset_id:<20} FOUND"
        )

        for path in found:

            print(
                f"  -> {path.name}"
            )

    else:

        print(
            f"{dataset_id:<20} NOT FOUND"
        )

print("=" * 90)

missing_datasets = []

for dataset_id, filenames in EXPECTED_DATASETS.items():

    if not any(
        (RAW_DIR / filename).exists()
        for filename in filenames
    ):

        missing_datasets.append(
            dataset_id
        )

# ------------------------------------------------------------
# Upload only missing datasets
# ------------------------------------------------------------

if missing_datasets:

    print(
        "\nThe following raw datasets are missing:"
    )

    for dataset_id in missing_datasets:

        print(
            f"  - {dataset_id}"
        )

    print(
        "\nUpload the corresponding original "
        "dataset files."
    )

    uploaded = files.upload()

    for filename in uploaded.keys():

        source = Path(
            filename
        )

        destination = (
            RAW_DIR /
            source.name
        )

        shutil.move(
            str(source),
            str(destination)
        )

        print(
            f"[SAVED] {destination}"
        )

else:

    print(
        "All required raw datasets are already available."
    )

print("=" * 90)
print("FINAL RAW DATASET INVENTORY")
print("=" * 90)

for file_path in sorted(
    RAW_DIR.iterdir()
):

    if file_path.is_file():

        print(
            f"{file_path.name:<45}"
            f"{file_path.stat().st_size / (1024**2):>10.2f} MB"
        )

print("=" * 90)

RAW DATASET VERIFICATION
adult_income         FOUND
  -> adult_income.csv
bank_marketing       FOUND
  -> bank_marketing.csv
diabetes_130us       FOUND
  -> diabetes_130us.csv
All required raw datasets are already available.
FINAL RAW DATASET INVENTORY
adult_income.csv                                   3.79 MB
bank_marketing.csv                                 4.40 MB
diabetes_130us.csv                                15.49 MB


In [3]:
# ============================================================
# 01.1 ENVIRONMENT, IMPORTS & PROJECT CONFIGURATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import json
import re
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

CONFIG_PATH = (
    PROJECT_ROOT /
    "config" /
    "air_llm_config.yaml"
)

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Master configuration not found:\n"
        f"{CONFIG_PATH}\n\n"
        "Run Notebook 00 successfully before "
        "running Notebook 01."
    )

# ------------------------------------------------------------
# LOAD MASTER CONFIGURATION
# ------------------------------------------------------------

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:
    CONFIG = yaml.safe_load(f)

DATASET_REGISTRY = CONFIG.get(
    "datasets",
    {}
)

print("=" * 90)
print("AIR-LLM | NOTEBOOK 01")
print("=" * 90)

print(
    f"Project root : {PROJECT_ROOT}"
)

print(
    f"Config       : {CONFIG_PATH}"
)

print(
    f"Datasets     : {list(DATASET_REGISTRY.keys())}"
)

AIR-LLM | NOTEBOOK 01
Project root : /content/drive/MyDrive/AIR_LLM_Research
Config       : /content/drive/MyDrive/AIR_LLM_Research/config/air_llm_config.yaml
Datasets     : ['adult_income', 'bank_marketing', 'diabetes_130us']


In [4]:
# ============================================================
# 01.2 RESEARCH DATASET CONFIGURATION
# ============================================================

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

DATASET_DISPLAY_NAMES = {
    "adult_income":
        "Adult Income",

    "bank_marketing":
        "Bank Marketing",

    "diabetes_130us":
        "Diabetes 130-US Hospitals",
}

# ------------------------------------------------------------
# TARGET REGISTRY
# ------------------------------------------------------------

TARGET_REGISTRY = {
    "adult_income":
        "income",

    "bank_marketing":
        "y",

    "diabetes_130us":
        "readmitted",
}

# ------------------------------------------------------------
# POSSIBLE SOURCE FILE NAMES
# ------------------------------------------------------------

DATASET_FILENAME_CANDIDATES = {

    "adult_income": [
        "adult_income.csv",
        "adult_income_processed.csv",
        "adult.csv",
        "adult_processed.csv",
        "adult-income.csv",
        "adult-income_processed.csv",
    ],

    "bank_marketing": [
        "bank_marketing.csv",
        "bank_marketing_processed.csv",
        "bank.csv",
        "bank_processed.csv",
        "bank-marketing.csv",
        "bank-marketing_processed.csv",
    ],

    "diabetes_130us": [
        "diabetes_130us.csv",
        "diabetes_130us_processed.csv",
        "diabetes.csv",
        "diabetes_processed.csv",
        "diabetes-130-us.csv",
    ],
}

SEARCH_ROOT = Path(
    "/content/drive/MyDrive"
)

assert set(DATASET_IDS) == set(
    TARGET_REGISTRY.keys()
)

print(
    "Dataset configuration validated."
)

Dataset configuration validated.


In [5]:
# ============================================================
# 01.3 DATASET DISCOVERY FUNCTION
# ============================================================

def discover_dataset_files(
    dataset_id,
    search_root=SEARCH_ROOT
):
    """
    Discover existing CSV files for a dataset.

    No files are created, moved, renamed, or modified.
    """

    filenames = DATASET_FILENAME_CANDIDATES[
        dataset_id
    ]

    candidates = []

    for filename in filenames:

        candidates.extend(
            search_root.rglob(filename)
        )

    # --------------------------------------------------------
    # Remove duplicate paths
    # --------------------------------------------------------

    candidates = sorted(
        set(candidates),
        key=lambda p: str(p)
    )

    # --------------------------------------------------------
    # Broad fallback search
    # --------------------------------------------------------

    if not candidates:

        keywords = {
            "adult_income":
                ["adult"],

            "bank_marketing":
                ["bank", "marketing"],

            "diabetes_130us":
                ["diabetes"],
        }

        possible = []

        for path in search_root.rglob("*"):

            if not path.is_file():
                continue

            if path.suffix.lower() != ".csv":
                continue

            name = path.name.lower()

            if any(
                keyword in name
                for keyword in keywords[dataset_id]
            ):
                possible.append(path)

        candidates = sorted(
            set(possible),
            key=lambda p: str(p)
        )

    return candidates

In [6]:
# ============================================================
# 01.4 DISCOVER ALL RESEARCH DATASETS
# ============================================================

DISCOVERED_DATASETS = {}

print("=" * 90)
print("DATASET DISCOVERY")
print("=" * 90)

for dataset_id in DATASET_IDS:

    candidates = discover_dataset_files(
        dataset_id
    )

    DISCOVERED_DATASETS[
        dataset_id
    ] = candidates

    print(
        f"\n{DATASET_DISPLAY_NAMES[dataset_id]}"
    )

    if candidates:

        for i, path in enumerate(
            candidates,
            start=1
        ):
            print(
                f"  {i}. {path}"
            )

    else:

        print(
            "  NOT FOUND"
        )

DATASET DISCOVERY

Adult Income
  1. /content/drive/MyDrive/AIR_LLM_Research/data/raw/adult_income.csv
  2. /content/drive/MyDrive/SPP_GAN_Project/datasets/processed/adult_income_processed.csv
  3. /content/drive/MyDrive/SPP_GAN_Project/datasets/raw/adult_income.csv

Bank Marketing
  1. /content/drive/MyDrive/AIR_LLM_Research/data/raw/bank_marketing.csv
  2. /content/drive/MyDrive/SPP_GAN_Project/datasets/processed/bank_marketing_processed.csv
  3. /content/drive/MyDrive/SPP_GAN_Project/datasets/raw/bank.csv
  4. /content/drive/MyDrive/SPP_GAN_Project/datasets/raw/bank_marketing.csv

Diabetes 130-US Hospitals
  1. /content/drive/MyDrive/AIR_LLM_Research/data/raw/diabetes_130us.csv
  2. /content/drive/MyDrive/SPP_GAN_Project/datasets/processed/diabetes_130us_processed.csv
  3. /content/drive/MyDrive/SPP_GAN_Project/datasets/raw/diabetes_130us.csv


In [7]:
# ============================================================
# 01.5 VALIDATE DATASET DISCOVERY
# ============================================================

missing_datasets = [
    dataset_id
    for dataset_id in DATASET_IDS
    if not DISCOVERED_DATASETS[dataset_id]
]

if missing_datasets:

    print("=" * 90)
    print("DATASET DISCOVERY FAILURE")
    print("=" * 90)

    for dataset_id in missing_datasets:

        print(
            f"\n{dataset_id}: NOT FOUND"
        )

    raise FileNotFoundError(
        "One or more required research datasets "
        "could not be located."
    )

print("=" * 90)
print("ALL REQUIRED DATASETS DISCOVERED")
print("=" * 90)

for dataset_id in DATASET_IDS:

    print(
        f"{dataset_id:<20}"
        f"{len(DISCOVERED_DATASETS[dataset_id])} "
        f"candidate file(s)"
    )

ALL REQUIRED DATASETS DISCOVERED
adult_income        3 candidate file(s)
bank_marketing      4 candidate file(s)
diabetes_130us      3 candidate file(s)


In [8]:
# ============================================================
# 01.6 SELECT DATASET FILES
# ============================================================

def select_dataset_file(
    dataset_id,
    candidates
):
    """
    Select the preferred dataset source.

    Priority:
        1. Exact raw filename
        2. Exact processed filename
        3. First discovered candidate
    """

    if not candidates:
        raise FileNotFoundError(
            f"No candidates for {dataset_id}"
        )

    # --------------------------------------------------------
    # Prefer exact raw filename
    # --------------------------------------------------------

    raw_names = [
        name
        for name in DATASET_FILENAME_CANDIDATES[
            dataset_id
        ]
        if "processed" not in name.lower()
    ]

    for path in candidates:

        if path.name in raw_names:

            return path, "raw"

    # --------------------------------------------------------
    # Otherwise use processed file
    # --------------------------------------------------------

    for path in candidates:

        if "processed" in path.name.lower():

            return path, "processed"

    # --------------------------------------------------------
    # Fallback
    # --------------------------------------------------------

    return candidates[0], "discovered"


SELECTED_DATASET_PATHS = {}
SELECTED_DATASET_SOURCE_TYPES = {}

for dataset_id in DATASET_IDS:

    path, source_type = select_dataset_file(
        dataset_id,
        DISCOVERED_DATASETS[dataset_id]
    )

    SELECTED_DATASET_PATHS[
        dataset_id
    ] = path

    SELECTED_DATASET_SOURCE_TYPES[
        dataset_id
    ] = source_type

print("=" * 90)
print("SELECTED DATASET SOURCES")
print("=" * 90)

for dataset_id in DATASET_IDS:

    print(
        f"\n{dataset_id}"
    )

    print(
        f"  Source type : "
        f"{SELECTED_DATASET_SOURCE_TYPES[dataset_id]}"
    )

    print(
        f"  Path        : "
        f"{SELECTED_DATASET_PATHS[dataset_id]}"
    )

SELECTED DATASET SOURCES

adult_income
  Source type : raw
  Path        : /content/drive/MyDrive/AIR_LLM_Research/data/raw/adult_income.csv

bank_marketing
  Source type : raw
  Path        : /content/drive/MyDrive/AIR_LLM_Research/data/raw/bank_marketing.csv

diabetes_130us
  Source type : raw
  Path        : /content/drive/MyDrive/AIR_LLM_Research/data/raw/diabetes_130us.csv


In [9]:
# ============================================================
# 01.7 LOAD ADULT INCOME — ROBUST HEADER HANDLING
# ============================================================

adult_income_path = (
    SELECTED_DATASET_PATHS["adult_income"]
)

adult_income_source = (
    SELECTED_DATASET_SOURCE_TYPES["adult_income"]
)

# ------------------------------------------------------------
# Detect whether the file already has a valid header
# ------------------------------------------------------------

adult_income_preview = pd.read_csv(
    adult_income_path,
    header=None,
    nrows=5,
    low_memory=False
)

# Standard UCI Adult dataset has 15 columns
ADULT_EXPECTED_COLUMNS = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income",
]

# ------------------------------------------------------------
# Determine whether first row is actually data
# ------------------------------------------------------------

first_row = (
    adult_income_preview.iloc[0]
    .astype(str)
    .str.strip()
    .str.lower()
    .tolist()
)

expected_header_normalized = [
    column.lower()
    for column in ADULT_EXPECTED_COLUMNS
]

has_expected_header = (
    first_row == expected_header_normalized
)

# ------------------------------------------------------------
# Load correctly
# ------------------------------------------------------------

if has_expected_header:

    adult_income_raw = pd.read_csv(
        adult_income_path,
        low_memory=False
    )

else:

    adult_income_raw = pd.read_csv(
        adult_income_path,
        header=None,
        names=ADULT_EXPECTED_COLUMNS,
        skipinitialspace=True,
        low_memory=False
    )

# ------------------------------------------------------------
# Clean whitespace from string values
# ------------------------------------------------------------

for column in adult_income_raw.select_dtypes(
    include=["object", "string"]
).columns:

    adult_income_raw[column] = (
        adult_income_raw[column]
        .astype("string")
        .str.strip()
    )

# ------------------------------------------------------------
# Validate structure
# ------------------------------------------------------------

if adult_income_raw.empty:
    raise ValueError(
        "Adult Income dataset contains zero rows."
    )

if list(adult_income_raw.columns) != ADULT_EXPECTED_COLUMNS:
    raise ValueError(
        "Adult Income columns do not match the "
        "expected UCI Adult structure.\n\n"
        f"Detected:\n{list(adult_income_raw.columns)}"
    )

print("=" * 90)
print("ADULT INCOME LOADED SUCCESSFULLY")
print("=" * 90)

print(
    f"Path       : {adult_income_path}"
)

print(
    f"Source     : {adult_income_source}"
)

print(
    f"Rows       : {len(adult_income_raw):,}"
)

print(
    f"Columns    : {adult_income_raw.shape[1]:,}"
)

print(
    f"Target     : {adult_income_raw.columns[-1]}"
)

print("=" * 90)

display(
    adult_income_raw.head()
)

ADULT INCOME LOADED SUCCESSFULLY
Path       : /content/drive/MyDrive/AIR_LLM_Research/data/raw/adult_income.csv
Source     : raw
Rows       : 32,561
Columns    : 15
Target     : income


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [10]:
# ============================================================
# 01.8 LOAD BANK MARKETING — ROBUST CSV INGESTION
# ============================================================

bank_marketing_path = (
    SELECTED_DATASET_PATHS["bank_marketing"]
)

bank_marketing_source = (
    SELECTED_DATASET_SOURCE_TYPES["bank_marketing"]
)

# ------------------------------------------------------------
# Detect delimiter from the first lines
# ------------------------------------------------------------

with open(
    bank_marketing_path,
    "r",
    encoding="utf-8",
    errors="ignore"
) as f:

    first_line = f.readline().strip()

# ------------------------------------------------------------
# Determine delimiter
# ------------------------------------------------------------

if first_line.count(";") > first_line.count(","):

    bank_delimiter = ";"

elif first_line.count(",") > 0:

    bank_delimiter = ","

else:

    bank_delimiter = ";"

print("=" * 90)
print("BANK MARKETING INGESTION")
print("=" * 90)

print(
    f"Detected delimiter : '{bank_delimiter}'"
)

# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------

bank_marketing_raw = pd.read_csv(
    bank_marketing_path,
    sep=bank_delimiter,
    low_memory=False
)

# ------------------------------------------------------------
# Remove accidental whitespace from column names
# ------------------------------------------------------------

bank_marketing_raw.columns = [
    str(column).strip()
    for column in bank_marketing_raw.columns
]

# ------------------------------------------------------------
# Remove whitespace from string values
# ------------------------------------------------------------

for column in bank_marketing_raw.select_dtypes(
    include=["object", "string"]
).columns:

    bank_marketing_raw[column] = (
        bank_marketing_raw[column]
        .astype("string")
        .str.strip()
    )

# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

if bank_marketing_raw.empty:

    raise ValueError(
        "Bank Marketing dataset contains zero rows."
    )

if bank_marketing_raw.shape[1] <= 1:

    raise ValueError(
        "Bank Marketing was loaded as one column. "
        "Delimiter detection failed."
    )

# ------------------------------------------------------------
# Validate target
# ------------------------------------------------------------

if "y" not in bank_marketing_raw.columns:

    raise KeyError(
        "Bank Marketing target column 'y' "
        "was not detected.\n\n"
        f"Detected columns:\n"
        f"{list(bank_marketing_raw.columns)}"
    )

print("=" * 90)
print("BANK MARKETING LOADED SUCCESSFULLY")
print("=" * 90)

print(
    f"Path       : {bank_marketing_path}"
)

print(
    f"Source     : {bank_marketing_source}"
)

print(
    f"Rows       : {len(bank_marketing_raw):,}"
)

print(
    f"Columns    : {bank_marketing_raw.shape[1]:,}"
)

print(
    f"Target     : y"
)

print("=" * 90)

display(
    bank_marketing_raw.head()
)

BANK MARKETING INGESTION
Detected delimiter : ';'
BANK MARKETING LOADED SUCCESSFULLY
Path       : /content/drive/MyDrive/AIR_LLM_Research/data/raw/bank_marketing.csv
Source     : raw
Rows       : 45,211
Columns    : 17
Target     : y


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [11]:
# ============================================================
# 01.9 LOAD DIABETES 130-US HOSPITALS
#      ROBUST HEADER + DELIMITER DETECTION
# ============================================================

diabetes_130us_path = (
    SELECTED_DATASET_PATHS["diabetes_130us"]
)

diabetes_130us_source = (
    SELECTED_DATASET_SOURCE_TYPES["diabetes_130us"]
)

# ------------------------------------------------------------
# EXPECTED UCI DIABETES 130-US COLUMNS
# ------------------------------------------------------------

DIABETES_EXPECTED_COLUMNS = [
    "encounter_id",
    "patient_nbr",
    "race",
    "gender",
    "age",
    "weight",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "payer_code",
    "medical_specialty",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "diag_1",
    "diag_2",
    "diag_3",
    "number_diagnoses",
    "max_glu_serum",
    "A1Cresult",
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
    "change",
    "diabetesMed",
    "readmitted",
]

# ------------------------------------------------------------
# READ PREVIEW
# ------------------------------------------------------------

with open(
    diabetes_130us_path,
    "r",
    encoding="utf-8",
    errors="ignore"
) as f:

    preview_lines = [
        f.readline().strip()
        for _ in range(5)
    ]

first_line = preview_lines[0]

# ------------------------------------------------------------
# DELIMITER DETECTION
# ------------------------------------------------------------

delimiter_candidates = {
    ",": first_line.count(","),
    ";": first_line.count(";"),
    "\t": first_line.count("\t"),
    "|": first_line.count("|"),
}

diabetes_delimiter = max(
    delimiter_candidates,
    key=delimiter_candidates.get
)

# ------------------------------------------------------------
# FALLBACK
# ------------------------------------------------------------

if (
    delimiter_candidates[
        diabetes_delimiter
    ] == 0
):

    diabetes_delimiter = ","

print("=" * 90)
print("DIABETES 130-US HOSPITALS INGESTION")
print("=" * 90)

print(
    f"Detected delimiter : "
    f"{repr(diabetes_delimiter)}"
)

# ------------------------------------------------------------
# FIRST ATTEMPT — NORMAL HEADER
# ------------------------------------------------------------

diabetes_test = pd.read_csv(
    diabetes_130us_path,
    sep=diabetes_delimiter,
    nrows=5,
    low_memory=False
)

# ------------------------------------------------------------
# DETERMINE WHETHER HEADER IS VALID
# ------------------------------------------------------------

test_columns = [
    str(column).strip()
    for column in diabetes_test.columns
]

normalized_test_columns = [
    column.lower()
    for column in test_columns
]

normalized_expected_columns = [
    column.lower()
    for column in DIABETES_EXPECTED_COLUMNS
]

header_matches_expected = (
    normalized_test_columns
    ==
    normalized_expected_columns
)

target_detected = (
    "readmitted"
    in normalized_test_columns
)

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

if header_matches_expected or target_detected:

    diabetes_130us_raw = pd.read_csv(
        diabetes_130us_path,
        sep=diabetes_delimiter,
        low_memory=False
    )

else:

    # --------------------------------------------------------
    # POSSIBLE HEADERLESS DATASET
    # --------------------------------------------------------

    diabetes_130us_raw = pd.read_csv(
        diabetes_130us_path,
        sep=diabetes_delimiter,
        header=None,
        low_memory=False
    )

    # --------------------------------------------------------
    # VALIDATE EXPECTED COLUMN COUNT
    # --------------------------------------------------------

    if (
        diabetes_130us_raw.shape[1]
        == len(DIABETES_EXPECTED_COLUMNS)
    ):

        diabetes_130us_raw.columns = (
            DIABETES_EXPECTED_COLUMNS
        )

    else:

        raise ValueError(
            "Unable to identify the Diabetes "
            "130-US Hospitals schema.\n\n"
            f"Detected columns: "
            f"{diabetes_130us_raw.shape[1]}\n"
            f"Expected columns: "
            f"{len(DIABETES_EXPECTED_COLUMNS)}\n\n"
            f"Detected header:\n"
            f"{test_columns}"
        )

# ------------------------------------------------------------
# STANDARDIZE COLUMN WHITESPACE ONLY
# ------------------------------------------------------------

diabetes_130us_raw.columns = [
    str(column).strip()
    for column in diabetes_130us_raw.columns
]

# ------------------------------------------------------------
# NORMALIZE TARGET COLUMN NAME IF NECESSARY
# ------------------------------------------------------------

target_candidates = [
    column
    for column in diabetes_130us_raw.columns
    if str(column).strip().lower()
    == "readmitted"
]

if target_candidates:

    target_column_actual = target_candidates[0]

    if target_column_actual != "readmitted":

        diabetes_130us_raw = (
            diabetes_130us_raw.rename(
                columns={
                    target_column_actual:
                    "readmitted"
                }
            )
        )

# ------------------------------------------------------------
# CLEAN STRING WHITESPACE
# ------------------------------------------------------------

for column in diabetes_130us_raw.select_dtypes(
    include=["object", "string"]
).columns:

    diabetes_130us_raw[column] = (
        diabetes_130us_raw[column]
        .astype("string")
        .str.strip()
    )

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

if diabetes_130us_raw.empty:

    raise ValueError(
        "Diabetes 130-US Hospitals dataset "
        "contains zero rows."
    )

if diabetes_130us_raw.shape[1] <= 1:

    raise ValueError(
        "Diabetes 130-US Hospitals dataset "
        "was loaded as one column. "
        "Delimiter detection failed."
    )

if "readmitted" not in (
    diabetes_130us_raw.columns
):

    raise KeyError(
        "Diabetes target 'readmitted' was not "
        "detected.\n\n"
        f"Detected columns:\n"
        f"{list(diabetes_130us_raw.columns)}"
    )

# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

print("=" * 90)
print("DIABETES 130-US HOSPITALS LOADED SUCCESSFULLY")
print("=" * 90)

print(
    f"Path       : {diabetes_130us_path}"
)

print(
    f"Source     : {diabetes_130us_source}"
)

print(
    f"Rows       : {len(diabetes_130us_raw):,}"
)

print(
    f"Columns    : {diabetes_130us_raw.shape[1]:,}"
)

print(
    f"Target     : readmitted"
)

print(
    f"Delimiter  : {repr(diabetes_delimiter)}"
)

print("=" * 90)

display(
    diabetes_130us_raw.head()
)

DIABETES 130-US HOSPITALS INGESTION
Detected delimiter : ','
DIABETES 130-US HOSPITALS LOADED SUCCESSFULLY
Path       : /content/drive/MyDrive/AIR_LLM_Research/data/raw/diabetes_130us.csv
Source     : raw
Rows       : 101,766
Columns    : 48
Target     : readmitted
Delimiter  : ','


,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,Caucasian,Female,[0-10),<NA>,6,25,1,1,<NA>,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),<NA>,1,1,7,3,<NA>,<NA>,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),<NA>,1,1,7,2,<NA>,<NA>,...,No,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),<NA>,1,1,7,2,<NA>,<NA>,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),<NA>,1,1,7,1,<NA>,<NA>,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [12]:
# ============================================================
# 01.10 DATASET IDENTIFICATION
# ============================================================

DATASETS = {

    "adult_income":
        adult_income_raw,

    "bank_marketing":
        bank_marketing_raw,

    "diabetes_130us":
        diabetes_130us_raw,
}

DATASET_PATHS = {

    "adult_income":
        adult_income_path,

    "bank_marketing":
        bank_marketing_path,

    "diabetes_130us":
        diabetes_130us_path,
}

DATASET_SOURCE_TYPES = {

    "adult_income":
        adult_income_source,

    "bank_marketing":
        bank_marketing_source,

    "diabetes_130us":
        diabetes_130us_source,
}

print("=" * 90)
print("DATASET OBJECTS REGISTERED")
print("=" * 90)

for dataset_id, df in DATASETS.items():

    print(
        f"{dataset_id:<20}"
        f"rows={df.shape[0]:,} | "
        f"columns={df.shape[1]:,}"
    )

DATASET OBJECTS REGISTERED
adult_income        rows=32,561 | columns=15
bank_marketing      rows=45,211 | columns=17
diabetes_130us      rows=101,766 | columns=48


In [13]:
# ============================================================
# 01.11 COLUMN NAME STANDARDIZATION
# ============================================================

def standardize_column_name(
    column_name
):
    """
    Convert column names to snake_case.
    """

    name = str(column_name).strip().lower()

    name = re.sub(
        r"[\s\-/]+",
        "_",
        name
    )

    name = re.sub(
        r"[^a-z0-9_]",
        "",
        name
    )

    name = re.sub(
        r"_+",
        "_",
        name
    )

    name = name.strip("_")

    return name


def standardize_columns(df):

    result = df.copy()

    original_columns = list(
        result.columns
    )

    standardized_columns = [
        standardize_column_name(column)
        for column in original_columns
    ]

    # --------------------------------------------------------
    # Ensure uniqueness
    # --------------------------------------------------------

    counts = {}
    unique_columns = []

    for column in standardized_columns:

        if column not in counts:

            counts[column] = 0
            unique_columns.append(
                column
            )

        else:

            counts[column] += 1

            unique_columns.append(
                f"{column}_{counts[column]}"
            )

    result.columns = unique_columns

    return (
        result,
        original_columns,
        unique_columns
    )


DATASETS_STANDARDIZED = {}

COLUMN_STANDARDIZATION_LOG = []

for dataset_id, df in DATASETS.items():

    (
        standardized_df,
        original_columns,
        standardized_columns
    ) = standardize_columns(df)

    DATASETS_STANDARDIZED[
        dataset_id
    ] = standardized_df

    for original, standardized in zip(
        original_columns,
        standardized_columns
    ):

        COLUMN_STANDARDIZATION_LOG.append({

            "dataset_id":
                dataset_id,

            "original_column":
                str(original),

            "standardized_column":
                str(standardized),

            "changed":
                bool(
                    str(original)
                    !=
                    str(standardized)
                ),
        })

COLUMN_STANDARDIZATION_DF = pd.DataFrame(
    COLUMN_STANDARDIZATION_LOG
)

print("=" * 90)
print("COLUMN STANDARDIZATION COMPLETED")
print("=" * 90)

for dataset_id, df in DATASETS_STANDARDIZED.items():

    print(
        f"{dataset_id:<20}"
        f"{df.shape[1]} columns | "
        f"unique={df.columns.is_unique}"
    )

COLUMN STANDARDIZATION COMPLETED
adult_income        15 columns | unique=True
bank_marketing      17 columns | unique=True
diabetes_130us      48 columns | unique=True


In [14]:
# ============================================================
# 01.12 TARGET IDENTIFICATION
# ============================================================

TARGET_IDENTIFICATION = []

for dataset_id, df in DATASETS_STANDARDIZED.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    if target not in df.columns:

        raise KeyError(
            f"\nTarget '{target}' was not found "
            f"in {dataset_id}.\n\n"
            f"Available columns:\n"
            f"{list(df.columns)}"
        )

    target_series = df[target]

    TARGET_IDENTIFICATION.append({

        "dataset_id":
            dataset_id,

        "target_column":
            target,

        "target_dtype":
            str(target_series.dtype),

        "target_missing_count":
            int(
                target_series.isna().sum()
            ),

        "target_missing_rate":
            float(
                target_series.isna().mean()
            ),

        "target_unique_values":
            int(
                target_series.nunique(
                    dropna=True
                )
            ),
    })

TARGET_IDENTIFICATION_DF = pd.DataFrame(
    TARGET_IDENTIFICATION
)

display(
    TARGET_IDENTIFICATION_DF
)

,dataset_id,target_column,target_dtype,target_missing_count,target_missing_rate,target_unique_values
0,adult_income,income,string,0,0.0,2
1,bank_marketing,y,string,0,0.0,2
2,diabetes_130us,readmitted,string,0,0.0,3


In [15]:
# ============================================================
# 01.13 TARGET DISTRIBUTION AUDIT
# ============================================================

TARGET_DISTRIBUTIONS = {}

for dataset_id, df in DATASETS_STANDARDIZED.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    distribution = (
        df[target]
        .value_counts(
            dropna=False
        )
        .rename_axis("value")
        .reset_index(
            name="count"
        )
    )

    distribution["percentage"] = (
        distribution["count"]
        / len(df)
        * 100.0
    )

    TARGET_DISTRIBUTIONS[
        dataset_id
    ] = distribution

    print("=" * 90)
    print(
        f"TARGET DISTRIBUTION | "
        f"{DATASET_DISPLAY_NAMES[dataset_id]}"
    )
    print("=" * 90)

    display(distribution)

TARGET DISTRIBUTION | Adult Income


,value,count,percentage
0,<=50K,24720,75.919044
1,>50K,7841,24.080956


TARGET DISTRIBUTION | Bank Marketing


,value,count,percentage
0,no,39922,88.30152
1,yes,5289,11.69848


TARGET DISTRIBUTION | Diabetes 130-US Hospitals


,value,count,percentage
0,NO,54864,53.911916
1,>30,35545,34.928169
2,<30,11357,11.159916


In [16]:
# ============================================================
# 01.14 DUPLICATE DETECTION
# ============================================================

DUPLICATE_AUDIT = []

for dataset_id, df in DATASETS_STANDARDIZED.items():

    duplicate_records = int(
        df.duplicated(
            keep="first"
        ).sum()
    )

    duplicate_all_occurrences = int(
        df.duplicated(
            keep=False
        ).sum()
    )

    duplicate_rate = (
        duplicate_records / len(df)
        if len(df) > 0
        else 0.0
    )

    DUPLICATE_AUDIT.append({

        "dataset_id":
            dataset_id,

        "total_rows":
            int(len(df)),

        "duplicate_records":
            duplicate_records,

        "duplicate_all_occurrences":
            duplicate_all_occurrences,

        "duplicate_rate":
            float(duplicate_rate),

        "unique_rows":
            int(
                df.drop_duplicates().shape[0]
            ),
    })

DUPLICATE_AUDIT_DF = pd.DataFrame(
    DUPLICATE_AUDIT
)

display(
    DUPLICATE_AUDIT_DF
)

,dataset_id,total_rows,duplicate_records,duplicate_all_occurrences,duplicate_rate,unique_rows
0,adult_income,32561,24,47,0.000737,32537
1,bank_marketing,45211,0,0,0.000000,45211
2,diabetes_130us,101766,0,0,0.000000,101766


In [17]:
# ============================================================
# 01.15 INITIAL MISSINGNESS AUDIT
# ============================================================

MISSING_VALUE_TOKENS = {
    "",
    " ",
    "na",
    "n/a",
    "nan",
    "null",
    "none",
    "?",
    "unknown",
    "not available",
    "not_applicable",
}

def audit_missingness(
    df,
    dataset_id
):

    records = []

    for column in df.columns:

        series = df[column]

        explicit_missing = int(
            series.isna().sum()
        )

        token_missing = 0

        if (
            pd.api.types.is_object_dtype(
                series
            )
            or
            pd.api.types.is_string_dtype(
                series
            )
        ):

            normalized = (
                series.astype("string")
                .str.strip()
                .str.lower()
            )

            token_missing = int(
                normalized.isin(
                    MISSING_VALUE_TOKENS
                ).sum()
            )

        records.append({

            "dataset_id":
                dataset_id,

            "column":
                column,

            "dtype":
                str(series.dtype),

            "rows":
                int(len(series)),

            "explicit_missing_count":
                explicit_missing,

            "explicit_missing_rate":
                float(
                    explicit_missing
                    / len(series)
                ),

            "missing_token_count":
                token_missing,

            "missing_token_rate":
                float(
                    token_missing
                    / len(series)
                ),

            "unique_values":
                int(
                    series.nunique(
                        dropna=True
                    )
                ),
        })

    return pd.DataFrame(records)


MISSINGNESS_AUDIT = {}

for dataset_id, df in DATASETS_STANDARDIZED.items():

    MISSINGNESS_AUDIT[
        dataset_id
    ] = audit_missingness(
        df,
        dataset_id
    )

INITIAL_MISSINGNESS_DF = pd.concat(
    MISSINGNESS_AUDIT.values(),
    ignore_index=True
)

print("=" * 90)
print("INITIAL MISSINGNESS AUDIT")
print("=" * 90)

display(
    INITIAL_MISSINGNESS_DF
)

INITIAL MISSINGNESS AUDIT


,dataset_id,column,dtype,rows,explicit_missing_count,explicit_missing_rate,missing_token_count,missing_token_rate,unique_values
0,adult_income,age,int64,32561,0,0.0,0,0.000000,73
1,adult_income,workclass,string,32561,0,0.0,1836,0.056386,9
2,adult_income,fnlwgt,int64,32561,0,0.0,0,0.000000,21648
3,adult_income,education,string,32561,0,0.0,0,0.000000,16
4,adult_income,education_num,int64,32561,0,0.0,0,0.000000,16
...,...,...,...,...,...,...,...,...,...
75,diabetes_130us,metformin_rosiglitazone,string,101766,0,0.0,0,0.000000,2
76,diabetes_130us,metformin_pioglitazone,string,101766,0,0.0,0,0.000000,2
77,diabetes_130us,change,string,101766,0,0.0,0,0.000000,2
78,diabetes_130us,diabetesmed,string,101766,0,0.0,0,0.000000,2


In [18]:
# ============================================================
# 01.16 DATASET-LEVEL MISSINGNESS SUMMARY
# ============================================================

MISSINGNESS_DATASET_SUMMARY = []

for dataset_id, df in DATASETS_STANDARDIZED.items():

    total_cells = (
        df.shape[0]
        *
        df.shape[1]
    )

    missing_cells = int(
        df.isna()
        .sum()
        .sum()
    )

    MISSINGNESS_DATASET_SUMMARY.append({

        "dataset_id":
            dataset_id,

        "rows":
            int(df.shape[0]),

        "columns":
            int(df.shape[1]),

        "total_cells":
            int(total_cells),

        "explicit_missing_cells":
            missing_cells,

        "explicit_missing_rate":
            float(
                missing_cells
                / total_cells
            ),

        "columns_with_missing":
            int(
                df.isna()
                .any()
                .sum()
            ),

        "rows_with_missing":
            int(
                df.isna()
                .any(axis=1)
                .sum()
            ),
    })

MISSINGNESS_DATASET_SUMMARY_DF = (
    pd.DataFrame(
        MISSINGNESS_DATASET_SUMMARY
    )
)

display(
    MISSINGNESS_DATASET_SUMMARY_DF
)

,dataset_id,rows,columns,total_cells,explicit_missing_cells,explicit_missing_rate,columns_with_missing,rows_with_missing
0,adult_income,32561,15,488415,0,0.000000,0,0
1,bank_marketing,45211,17,768587,0,0.000000,0,0
2,diabetes_130us,101766,48,4884768,374017,0.076568,9,101766


In [19]:
# ============================================================
# 01.17 DATA-TYPE AUDIT
# ============================================================

def infer_feature_type(series):

    if pd.api.types.is_bool_dtype(series):
        return "boolean"

    if pd.api.types.is_numeric_dtype(series):
        return "numerical"

    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"

    return "categorical"


DTYPE_AUDIT = []

for dataset_id, df in DATASETS_STANDARDIZED.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    for column in df.columns:

        series = df[column]

        DTYPE_AUDIT.append({

            "dataset_id":
                dataset_id,

            "column":
                column,

            "pandas_dtype":
                str(series.dtype),

            "inferred_type":
                infer_feature_type(
                    series
                ),

            "is_target":
                bool(
                    column == target
                ),

            "unique_values":
                int(
                    series.nunique(
                        dropna=True
                    )
                ),

            "missing_count":
                int(
                    series.isna().sum()
                ),

            "missing_rate":
                float(
                    series.isna().mean()
                ),
        })

DTYPE_AUDIT_DF = pd.DataFrame(
    DTYPE_AUDIT
)

display(
    DTYPE_AUDIT_DF
)

,dataset_id,column,pandas_dtype,inferred_type,is_target,unique_values,missing_count,missing_rate
0,adult_income,age,int64,numerical,False,73,0,0.0
1,adult_income,workclass,string,categorical,False,9,0,0.0
2,adult_income,fnlwgt,int64,numerical,False,21648,0,0.0
3,adult_income,education,string,categorical,False,16,0,0.0
4,adult_income,education_num,int64,numerical,False,16,0,0.0
...,...,...,...,...,...,...,...,...
75,diabetes_130us,metformin_rosiglitazone,string,categorical,False,2,0,0.0
76,diabetes_130us,metformin_pioglitazone,string,categorical,False,2,0,0.0
77,diabetes_130us,change,string,categorical,False,2,0,0.0
78,diabetes_130us,diabetesmed,string,categorical,False,2,0,0.0


In [20]:
# ============================================================
# 01.18 FEATURE TYPE SUMMARY
# ============================================================

FEATURE_TYPE_SUMMARY = (
    DTYPE_AUDIT_DF
    .groupby(
        [
            "dataset_id",
            "inferred_type"
        ]
    )
    .size()
    .reset_index(
        name="feature_count"
    )
)

display(
    FEATURE_TYPE_SUMMARY
)

,dataset_id,inferred_type,feature_count
0,adult_income,categorical,9
1,adult_income,numerical,6
2,bank_marketing,categorical,10
3,bank_marketing,numerical,7
4,diabetes_130us,categorical,37
5,diabetes_130us,numerical,11


In [21]:
# ============================================================
# 01.19 COMPLETE DATASET SUMMARY
# ============================================================

DATASET_SUMMARY = []

for dataset_id, df in DATASETS_STANDARDIZED.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    numerical_features = sum(
        pd.api.types.is_numeric_dtype(
            df[column]
        )
        for column in df.columns
        if column != target
    )

    categorical_features = (
        df.shape[1]
        - 1
        - numerical_features
    )

    total_cells = (
        df.shape[0]
        *
        df.shape[1]
    )

    missing_cells = int(
        df.isna()
        .sum()
        .sum()
    )

    DATASET_SUMMARY.append({

        "dataset_id":
            dataset_id,

        "dataset_name":
            DATASET_DISPLAY_NAMES[
                dataset_id
            ],

        "source_type":
            DATASET_SOURCE_TYPES[
                dataset_id
            ],

        "rows":
            int(df.shape[0]),

        "columns":
            int(df.shape[1]),

        "numerical_features":
            int(numerical_features),

        "categorical_features":
            int(categorical_features),

        "target":
            target,

        "target_unique_values":
            int(
                df[target]
                .nunique(
                    dropna=True
                )
            ),

        "duplicate_records":
            int(
                df.duplicated(
                    keep="first"
                ).sum()
            ),

        "columns_with_missing":
            int(
                df.isna()
                .any()
                .sum()
            ),

        "missing_cells":
            missing_cells,

        "overall_missing_rate":
            float(
                missing_cells
                / total_cells
            ),

        "memory_mb":
            float(
                df.memory_usage(
                    deep=True
                ).sum()
                / 1024**2
            ),
    })

DATASET_SUMMARY_DF = pd.DataFrame(
    DATASET_SUMMARY
)

print("=" * 90)
print("COMPLETE DATASET SUMMARY")
print("=" * 90)

display(
    DATASET_SUMMARY_DF
)

COMPLETE DATASET SUMMARY


,dataset_id,dataset_name,source_type,rows,columns,numerical_features,categorical_features,target,target_unique_values,duplicate_records,columns_with_missing,missing_cells,overall_missing_rate,memory_mb
0,adult_income,Adult Income,raw,32561,15,6,8,income,2,24,0,0,0.000000,17.646922
1,bank_marketing,Bank Marketing,raw,45211,17,7,9,y,2,0,0,0,0.000000,25.748668
2,diabetes_130us,Diabetes 130-US Hospitals,raw,101766,48,11,36,readmitted,3,0,9,374017,0.076568,196.568024


In [22]:
# ============================================================
# 01.20 SAVE DATASET REGISTRY & AUDIT ARTIFACTS
#      SELF-CONTAINED PUBLICATION VERSION
# ============================================================

from pathlib import Path
import pandas as pd
import yaml
import json

# ------------------------------------------------------------
# DIRECTORIES
# ------------------------------------------------------------

REGISTRY_DIR = (
    PROJECT_ROOT /
    "config"
)

AUDIT_DIR = (
    PROJECT_ROOT /
    "results" /
    "raw"
)

TABLE_DIR = (
    PROJECT_ROOT /
    "results" /
    "tables"
)

REGISTRY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# 1. REBUILD DATASET IDENTIFICATION
# ============================================================

DATASET_IDENTIFICATION = []

for dataset_id, df in DATASETS_STANDARDIZED.items():

    DATASET_IDENTIFICATION.append({

        "dataset_id":
            dataset_id,

        "dataset_name":
            DATASET_DISPLAY_NAMES[
                dataset_id
            ],

        "source_type":
            DATASET_SOURCE_TYPES[
                dataset_id
            ],

        "source_path":
            str(
                DATASET_PATHS[
                    dataset_id
                ]
            ),

        "rows":
            int(df.shape[0]),

        "columns":
            int(df.shape[1]),

        "memory_mb":
            float(
                df.memory_usage(
                    deep=True
                ).sum()
                / 1024**2
            ),
    })

DATASET_IDENTIFICATION_DF = pd.DataFrame(
    DATASET_IDENTIFICATION
)

# ============================================================
# 2. REBUILD COLUMN STANDARDIZATION AUDIT
# ============================================================

COLUMN_STANDARDIZATION_LOG = []

for dataset_id, df in DATASETS.items():

    standardized_df = (
        DATASETS_STANDARDIZED[
            dataset_id
        ]
    )

    original_columns = list(
        df.columns
    )

    standardized_columns = list(
        standardized_df.columns
    )

    for original, standardized in zip(
        original_columns,
        standardized_columns
    ):

        COLUMN_STANDARDIZATION_LOG.append({

            "dataset_id":
                dataset_id,

            "original_column":
                str(original),

            "standardized_column":
                str(standardized),

            "changed":
                bool(
                    str(original)
                    != str(standardized)
                ),
        })

COLUMN_STANDARDIZATION_DF = pd.DataFrame(
    COLUMN_STANDARDIZATION_LOG
)

# ============================================================
# 3. REBUILD TARGET IDENTIFICATION
# ============================================================

TARGET_IDENTIFICATION = []

for dataset_id, df in DATASETS_STANDARDIZED.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    if target not in df.columns:

        raise KeyError(
            f"Target '{target}' not found "
            f"in {dataset_id}."
        )

    target_series = df[target]

    TARGET_IDENTIFICATION.append({

        "dataset_id":
            dataset_id,

        "target_column":
            target,

        "target_dtype":
            str(
                target_series.dtype
            ),

        "target_missing_count":
            int(
                target_series.isna().sum()
            ),

        "target_missing_rate":
            float(
                target_series.isna().mean()
            ),

        "target_unique_values":
            int(
                target_series.nunique(
                    dropna=True
                )
            ),
    })

TARGET_IDENTIFICATION_DF = pd.DataFrame(
    TARGET_IDENTIFICATION
)

# ============================================================
# 4. REBUILD DUPLICATE AUDIT
# ============================================================

DUPLICATE_AUDIT = []

for dataset_id, df in DATASETS_STANDARDIZED.items():

    duplicate_records = int(
        df.duplicated(
            keep="first"
        ).sum()
    )

    duplicate_all = int(
        df.duplicated(
            keep=False
        ).sum()
    )

    DUPLICATE_AUDIT.append({

        "dataset_id":
            dataset_id,

        "total_rows":
            int(len(df)),

        "duplicate_records":
            duplicate_records,

        "duplicate_all_occurrences":
            duplicate_all,

        "duplicate_rate":
            float(
                duplicate_records /
                len(df)
            ),

        "unique_rows":
            int(
                df.drop_duplicates().shape[0]
            ),
    })

DUPLICATE_AUDIT_DF = pd.DataFrame(
    DUPLICATE_AUDIT
)

# ============================================================
# 5. REBUILD MISSINGNESS AUDIT
# ============================================================

MISSINGNESS_AUDIT = []

MISSING_VALUE_TOKENS = {
    "",
    " ",
    "na",
    "n/a",
    "nan",
    "null",
    "none",
    "?",
    "unknown",
    "not available",
    "not_applicable",
}

for dataset_id, df in DATASETS_STANDARDIZED.items():

    for column in df.columns:

        series = df[column]

        explicit_missing = int(
            series.isna().sum()
        )

        token_missing = 0

        if (
            pd.api.types.is_object_dtype(
                series
            )
            or
            pd.api.types.is_string_dtype(
                series
            )
        ):

            normalized = (
                series.astype("string")
                .str.strip()
                .str.lower()
            )

            token_missing = int(
                normalized.isin(
                    MISSING_VALUE_TOKENS
                ).sum()
            )

        MISSINGNESS_AUDIT.append({

            "dataset_id":
                dataset_id,

            "column":
                column,

            "dtype":
                str(series.dtype),

            "rows":
                int(len(series)),

            "explicit_missing_count":
                explicit_missing,

            "explicit_missing_rate":
                float(
                    explicit_missing /
                    len(series)
                ),

            "missing_token_count":
                token_missing,

            "missing_token_rate":
                float(
                    token_missing /
                    len(series)
                ),

            "unique_values":
                int(
                    series.nunique(
                        dropna=True
                    )
                ),
        })

INITIAL_MISSINGNESS_DF = pd.DataFrame(
    MISSINGNESS_AUDIT
)

# ============================================================
# 6. REBUILD DATA-TYPE AUDIT
# ============================================================

def infer_feature_type(series):

    if pd.api.types.is_bool_dtype(series):
        return "boolean"

    if pd.api.types.is_numeric_dtype(series):
        return "numerical"

    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"

    return "categorical"


DTYPE_AUDIT = []

for dataset_id, df in DATASETS_STANDARDIZED.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    for column in df.columns:

        series = df[column]

        DTYPE_AUDIT.append({

            "dataset_id":
                dataset_id,

            "column":
                column,

            "pandas_dtype":
                str(series.dtype),

            "inferred_type":
                infer_feature_type(
                    series
                ),

            "is_target":
                bool(
                    column == target
                ),

            "unique_values":
                int(
                    series.nunique(
                        dropna=True
                    )
                ),

            "missing_count":
                int(
                    series.isna().sum()
                ),

            "missing_rate":
                float(
                    series.isna().mean()
                ),
        })

DTYPE_AUDIT_DF = pd.DataFrame(
    DTYPE_AUDIT
)

# ============================================================
# 7. FEATURE TYPE SUMMARY
# ============================================================

FEATURE_TYPE_SUMMARY = (
    DTYPE_AUDIT_DF
    .groupby(
        [
            "dataset_id",
            "inferred_type"
        ]
    )
    .size()
    .reset_index(
        name="feature_count"
    )
)

# ============================================================
# 8. DATASET-LEVEL MISSINGNESS SUMMARY
# ============================================================

MISSINGNESS_DATASET_SUMMARY = []

for dataset_id, df in DATASETS_STANDARDIZED.items():

    total_cells = (
        df.shape[0]
        *
        df.shape[1]
    )

    missing_cells = int(
        df.isna()
        .sum()
        .sum()
    )

    MISSINGNESS_DATASET_SUMMARY.append({

        "dataset_id":
            dataset_id,

        "rows":
            int(df.shape[0]),

        "columns":
            int(df.shape[1]),

        "total_cells":
            int(total_cells),

        "explicit_missing_cells":
            missing_cells,

        "explicit_missing_rate":
            float(
                missing_cells /
                total_cells
            ),

        "columns_with_missing":
            int(
                df.isna()
                .any()
                .sum()
            ),

        "rows_with_missing":
            int(
                df.isna()
                .any(axis=1)
                .sum()
            ),
    })

MISSINGNESS_DATASET_SUMMARY_DF = (
    pd.DataFrame(
        MISSINGNESS_DATASET_SUMMARY
    )
)

# ============================================================
# 9. COMPLETE DATASET SUMMARY
# ============================================================

DATASET_SUMMARY = []

for dataset_id, df in DATASETS_STANDARDIZED.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    numerical_features = sum(
        pd.api.types.is_numeric_dtype(
            df[column]
        )
        for column in df.columns
        if column != target
    )

    categorical_features = (
        df.shape[1]
        - 1
        - numerical_features
    )

    total_cells = (
        df.shape[0]
        *
        df.shape[1]
    )

    missing_cells = int(
        df.isna()
        .sum()
        .sum()
    )

    DATASET_SUMMARY.append({

        "dataset_id":
            dataset_id,

        "dataset_name":
            DATASET_DISPLAY_NAMES[
                dataset_id
            ],

        "source_type":
            DATASET_SOURCE_TYPES[
                dataset_id
            ],

        "rows":
            int(df.shape[0]),

        "columns":
            int(df.shape[1]),

        "numerical_features":
            int(numerical_features),

        "categorical_features":
            int(categorical_features),

        "target":
            target,

        "target_unique_values":
            int(
                df[target]
                .nunique(
                    dropna=True
                )
            ),

        "duplicate_records":
            int(
                df.duplicated(
                    keep="first"
                ).sum()
            ),

        "columns_with_missing":
            int(
                df.isna()
                .any()
                .sum()
            ),

        "missing_cells":
            missing_cells,

        "overall_missing_rate":
            float(
                missing_cells /
                total_cells
            ),

        "memory_mb":
            float(
                df.memory_usage(
                    deep=True
                ).sum()
                / 1024**2
            ),
    })

DATASET_SUMMARY_DF = pd.DataFrame(
    DATASET_SUMMARY
)

# ============================================================
# 10. RAW DATASET REGISTRY
# ============================================================

RAW_DATASET_REGISTRY = {}

for dataset_id, df in DATASETS_STANDARDIZED.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    RAW_DATASET_REGISTRY[
        dataset_id
    ] = {

        "dataset_id":
            dataset_id,

        "dataset_name":
            DATASET_DISPLAY_NAMES[
                dataset_id
            ],

        "source_path":
            str(
                DATASET_PATHS[
                    dataset_id
                ]
            ),

        "source_type":
            DATASET_SOURCE_TYPES[
                dataset_id
            ],

        "rows":
            int(df.shape[0]),

        "columns":
            int(df.shape[1]),

        "column_names":
            [
                str(column)
                for column in df.columns
            ],

        "target_column":
            target,

        "target_dtype":
            str(
                df[target].dtype
            ),

        "duplicate_records":
            int(
                df.duplicated(
                    keep="first"
                ).sum()
            ),

        "missing_cells":
            int(
                df.isna()
                .sum()
                .sum()
            ),

        "missing_rate":
            float(
                df.isna()
                .sum()
                .sum()
                /
                (
                    df.shape[0]
                    *
                    df.shape[1]
                )
            ),
    }

# ============================================================
# 11. SAVE MASTER RAW REGISTRY
# ============================================================

RAW_REGISTRY_PATH = (
    REGISTRY_DIR /
    "raw_dataset_registry.yaml"
)

with open(
    RAW_REGISTRY_PATH,
    "w",
    encoding="utf-8"
) as f:

    yaml.safe_dump(
        RAW_DATASET_REGISTRY,
        f,
        sort_keys=False,
        allow_unicode=True
    )

# ============================================================
# 12. SAVE AUDIT TABLES
# ============================================================

AUDIT_TABLES = {

    "dataset_identification.csv":
        DATASET_IDENTIFICATION_DF,

    "column_standardization_audit.csv":
        COLUMN_STANDARDIZATION_DF,

    "target_identification.csv":
        TARGET_IDENTIFICATION_DF,

    "duplicate_audit.csv":
        DUPLICATE_AUDIT_DF,

    "initial_missingness_audit.csv":
        INITIAL_MISSINGNESS_DF,

    "dataset_missingness_summary.csv":
        MISSINGNESS_DATASET_SUMMARY_DF,

    "dtype_audit.csv":
        DTYPE_AUDIT_DF,

    "feature_type_summary.csv":
        FEATURE_TYPE_SUMMARY,

    "dataset_summary.csv":
        DATASET_SUMMARY_DF,
}

for filename, dataframe in AUDIT_TABLES.items():

    dataframe.to_csv(
        AUDIT_DIR / filename,
        index=False
    )

# ============================================================
# 13. SAVE TARGET DISTRIBUTIONS
# ============================================================

for dataset_id, df in DATASETS_STANDARDIZED.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    distribution = (
        df[target]
        .value_counts(
            dropna=False
        )
        .rename_axis("value")
        .reset_index(
            name="count"
        )
    )

    distribution["percentage"] = (
        distribution["count"]
        / len(df)
        * 100.0
    )

    distribution.to_csv(
        AUDIT_DIR /
        f"{dataset_id}_target_distribution.csv",
        index=False
    )

# ============================================================
# 14. FINAL SAVE REPORT
# ============================================================

print("=" * 90)
print("NOTEBOOK 01 ARTIFACTS SAVED SUCCESSFULLY")
print("=" * 90)

print(
    f"Registry : {RAW_REGISTRY_PATH}"
)

print(
    f"Audits   : {AUDIT_DIR}"
)

print(
    f"Tables   : {TABLE_DIR}"
)

print(
    f"\nDatasets saved : "
    f"{len(DATASETS_STANDARDIZED)}"
)

print(
    f"Audit tables  : "
    f"{len(AUDIT_TABLES)}"
)

print("=" * 90)

NOTEBOOK 01 ARTIFACTS SAVED SUCCESSFULLY
Registry : /content/drive/MyDrive/AIR_LLM_Research/config/raw_dataset_registry.yaml
Audits   : /content/drive/MyDrive/AIR_LLM_Research/results/raw
Tables   : /content/drive/MyDrive/AIR_LLM_Research/results/tables

Datasets saved : 3
Audit tables  : 9


In [23]:
# ============================================================
# 01.21 FINAL VALIDATION
# ============================================================

from pathlib import Path
import yaml
import pandas as pd

print("=" * 90)
print("AIR-LLM | NOTEBOOK 01 FINAL VALIDATION")
print("=" * 90)

# ------------------------------------------------------------
# EXPECTED DATASETS
# ------------------------------------------------------------

EXPECTED_DATASETS = {
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
}

# ------------------------------------------------------------
# 1. DATASET OBJECT VALIDATION
# ------------------------------------------------------------

assert set(
    DATASETS_STANDARDIZED.keys()
) == EXPECTED_DATASETS, (
    "Dataset registry does not contain "
    "the expected three datasets."
)

# ------------------------------------------------------------
# 2. DATASET STRUCTURE VALIDATION
# ------------------------------------------------------------

for dataset_id, df in DATASETS_STANDARDIZED.items():

    assert len(df) > 0, (
        f"{dataset_id} contains zero rows."
    )

    assert df.shape[1] > 1, (
        f"{dataset_id} contains only one column."
    )

    assert df.columns.is_unique, (
        f"{dataset_id} contains duplicate column names."
    )

    target = TARGET_REGISTRY[
        dataset_id
    ]

    assert target in df.columns, (
        f"Target '{target}' is missing "
        f"from {dataset_id}."
    )

# ------------------------------------------------------------
# 3. RAW REGISTRY VALIDATION
# ------------------------------------------------------------

assert RAW_REGISTRY_PATH.exists(), (
    "Raw dataset registry was not saved."
)

with open(
    RAW_REGISTRY_PATH,
    "r",
    encoding="utf-8"
) as f:

    VALIDATED_REGISTRY = yaml.safe_load(f)

assert isinstance(
    VALIDATED_REGISTRY,
    dict
), "Raw dataset registry is not a valid dictionary."

assert set(
    VALIDATED_REGISTRY.keys()
) == EXPECTED_DATASETS, (
    "Saved registry does not contain "
    "the expected datasets."
)

# ------------------------------------------------------------
# 4. AUDIT FILE VALIDATION
# ------------------------------------------------------------

EXPECTED_AUDIT_FILES = [

    "dataset_identification.csv",

    "column_standardization_audit.csv",

    "target_identification.csv",

    "duplicate_audit.csv",

    "initial_missingness_audit.csv",

    "dataset_missingness_summary.csv",

    "dtype_audit.csv",

    "feature_type_summary.csv",

    "dataset_summary.csv",
]

for filename in EXPECTED_AUDIT_FILES:

    file_path = (
        AUDIT_DIR /
        filename
    )

    assert file_path.exists(), (
        f"Missing audit file:\n{file_path}"
    )

    # --------------------------------------------------------
    # Verify that the CSV is readable
    # --------------------------------------------------------

    validation_df = pd.read_csv(
        file_path
    )

    assert not validation_df.empty, (
        f"Audit file is empty:\n{file_path}"
    )

# ------------------------------------------------------------
# 5. TARGET DISTRIBUTION VALIDATION
# ------------------------------------------------------------

for dataset_id in EXPECTED_DATASETS:

    target_distribution_path = (
        AUDIT_DIR /
        f"{dataset_id}_target_distribution.csv"
    )

    assert target_distribution_path.exists(), (
        "Missing target distribution file:\n"
        f"{target_distribution_path}"
    )

    target_distribution = pd.read_csv(
        target_distribution_path
    )

    assert not target_distribution.empty, (
        f"Target distribution is empty:\n"
        f"{target_distribution_path}"
    )

# ------------------------------------------------------------
# 6. DATASET SUMMARY VALIDATION
# ------------------------------------------------------------

assert (
    len(DATASET_SUMMARY_DF)
    == len(EXPECTED_DATASETS)
), (
    "Dataset summary does not contain "
    "all expected datasets."
)

# ------------------------------------------------------------
# 7. TARGET IDENTIFICATION VALIDATION
# ------------------------------------------------------------

assert (
    len(TARGET_IDENTIFICATION_DF)
    == len(EXPECTED_DATASETS)
), (
    "Target identification does not contain "
    "all expected datasets."
)

for dataset_id in EXPECTED_DATASETS:

    target = TARGET_REGISTRY[
        dataset_id
    ]

    detected_target = (
        TARGET_IDENTIFICATION_DF.loc[
            TARGET_IDENTIFICATION_DF[
                "dataset_id"
            ] == dataset_id,
            "target_column"
        ]
        .iloc[0]
    )

    assert detected_target == target, (
        f"Target mismatch for "
        f"{dataset_id}: "
        f"expected={target}, "
        f"detected={detected_target}"
    )

# ------------------------------------------------------------
# 8. FINAL DATASET REPORT
# ------------------------------------------------------------

print("\nDATASET VALIDATION")
print("-" * 90)

for dataset_id, df in (
    DATASETS_STANDARDIZED.items()
):

    target = TARGET_REGISTRY[
        dataset_id
    ]

    print(
        f"{dataset_id:<20}"
        f"rows={len(df):>10,} | "
        f"columns={df.shape[1]:>3} | "
        f"target={target}"
    )

# ------------------------------------------------------------
# 9. FILE REPORT
# ------------------------------------------------------------

print("\nSAVED ARTIFACTS")
print("-" * 90)

print(
    f"Registry : {RAW_REGISTRY_PATH}"
)

print(
    f"Audits   : {AUDIT_DIR}"
)

print(
    f"Audit files : "
    f"{len(EXPECTED_AUDIT_FILES)}"
)

# ------------------------------------------------------------
# 10. FINAL STATUS
# ------------------------------------------------------------

print("\nVALIDATION STATUS")
print("-" * 90)

print(
    "Dataset registry     : PASSED"
)

print(
    "Dataset structure    : PASSED"
)

print(
    "Target identification: PASSED"
)

print(
    "Audit artifacts      : PASSED"
)

print(
    "Target distributions : PASSED"
)

print(
    "Registry reload      : PASSED"
)

print("=" * 90)
print("NOTEBOOK 01 COMPLETE — VALIDATION PASSED")
print("=" * 90)

AIR-LLM | NOTEBOOK 01 FINAL VALIDATION

DATASET VALIDATION
------------------------------------------------------------------------------------------
adult_income        rows=    32,561 | columns= 15 | target=income
bank_marketing      rows=    45,211 | columns= 17 | target=y
diabetes_130us      rows=   101,766 | columns= 48 | target=readmitted

SAVED ARTIFACTS
------------------------------------------------------------------------------------------
Registry : /content/drive/MyDrive/AIR_LLM_Research/config/raw_dataset_registry.yaml
Audits   : /content/drive/MyDrive/AIR_LLM_Research/results/raw
Audit files : 9

VALIDATION STATUS
------------------------------------------------------------------------------------------
Dataset registry     : PASSED
Dataset structure    : PASSED
Target identification: PASSED
Audit artifacts      : PASSED
Target distributions : PASSED
Registry reload      : PASSED
NOTEBOOK 01 COMPLETE — VALIDATION PASSED
